In [373]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [394]:
xx=pd.read_csv('/kaggle/input/datasets/ishan8061/incomeclassification/train.csv')
xx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43957 entries, 0 to 43956
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              43957 non-null  int64 
 1   workclass        41459 non-null  object
 2   fnlwgt           43957 non-null  int64 
 3   education        43957 non-null  object
 4   educational-num  43957 non-null  int64 
 5   marital-status   43957 non-null  object
 6   occupation       41451 non-null  object
 7   relationship     43957 non-null  object
 8   race             43957 non-null  object
 9   gender           43957 non-null  object
 10  capital-gain     43957 non-null  int64 
 11  capital-loss     43957 non-null  int64 
 12  hours-per-week   43957 non-null  int64 
 13  native-country   43194 non-null  object
 14  income_>50K      43957 non-null  int64 
dtypes: int64(7), object(8)
memory usage: 5.0+ MB


In [374]:
train=pd.read_csv('/kaggle/input/datasets/ishan8061/incomeclassification/train.csv')
test=pd.read_csv('/kaggle/input/datasets/ishan8061/incomeclassification/test.csv')

In [375]:
train=train.rename(columns={'income_>50K': '>50k'})

In [376]:
values=['Bachelors', 'Masters', 'Prof-school', 'Assoc-acdm', 'Assoc-voc']
train['IsHighEducated']=np.where(train['education'].isin(values), True, False)

In [377]:
#pd.crosstab(train['workclass'], train['>50k'], normalize='index')*100
values=['Federal-gov', 'Self-emp-inc']
train['GovernmentOrBoss']=np.where(train['workclass'].isin(values), True, False)

In [378]:
#pd.crosstab(train['marital-status'], train['>50k'], normalize='index')*100
values=['Divorced', 'Never-married', 'Separated', 'Widowed']
train['Married']=np.where(train['marital-status'].isin(values), False, True)

In [379]:
#pd.crosstab(train['occupation'], train['>50k'], normalize='index')*100
values=['Prof-specialty', 'Exec-managerial']
train['HighPaying']=np.where(train['occupation'].isin(values), True, False)

In [380]:
values=['Asian-Pac-Islander', 'White']
train['AsianOrWhite']=np.where(train['race'].isin(values), True, False)

In [381]:
Europe=['Portugal', 'Italy', 'Germany', 'France', 'Greece', 'Yugoslavia', 'Poland', 'Hungary']
asian=['Japan', 'Vietnam', 'Thailand', 'India', 'Cambodia', 'Iran', 'Taiwan', 'Hong', 'China', 'Philippines']
GB=['England', 'Scotland', 'Ireland']
america=['Mexico', 'Canada', 'United-States']

train['Country']='Others'
train['Country']=np.where(train['native-country'].isin(Europe), 'Europe', train['Country'])
train['Country']=np.where(train['native-country'].isin(asian), 'Asia', train['Country'])
train['Country']=np.where(train['native-country'].isin(GB), 'GB', train['Country'])
train['Country']=np.where(train['native-country'].isin(america), 'North America', train['Country'])

In [382]:
train['profit']=train['capital-gain']-train['capital-loss']
train['HighProfit']=np.where(train['profit']>10000, True, False)

In [383]:
train['AgeBin']='Others'
train['AgeBin']=np.where((train['age']>=0) & (train['age']<=10), 'Child', train['AgeBin'])
train['AgeBin']=np.where((train['age']>10) & (train['age']<=20), 'Teenage', train['AgeBin'])
train['AgeBin']=np.where((train['age']>20) & (train['age']<=30), 'Youth', train['AgeBin'])
train['AgeBin']=np.where((train['age']>30) & (train['age']<=55), 'Adult', train['AgeBin'])
train['AgeBin']=np.where((train['age']>55) & (train['age']<=80), 'Retired', train['AgeBin'])
train['AgeBin']=np.where((train['age']>80) & (train['age']<=100), 'Senile', train['AgeBin'])

In [385]:
features=['workclass', 'occupation', 'education', 'marital-status', 'gender', 'relationship', 'IsHighEducated', 'GovernmentOrBoss', 'Married', \
         'HighPaying', 'AsianOrWhite', 'Country', 'HighProfit', 'AgeBin']

In [386]:
for i in features:
    train[i]=train[i].astype('category')

In [387]:
X=train[features]
y=train['>50k']

x_train, x_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [399]:
model=lgb.LGBMClassifier(
    num_leaves=31,
    n_estimators=150,
    learning_rate=0.05,
    boosting_type='gbdt',
    random_state=42
)
model.fit(x_train, y_train)
preds=model.predict(x_test)
acc=accuracy_score(y_test, preds)
print(acc)

[LightGBM] [Info] Number of positive: 8414, number of negative: 26751
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 85
[LightGBM] [Info] Number of data points in the train set: 35165, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239272 -> initscore=-1.156675
[LightGBM] [Info] Start training from score -1.156675
0.847361237488626


# TEST

In [389]:
test=test.rename(columns={'income_>50K': '>50k'})

values=['Bachelors', 'Masters', 'Prof-school', 'Assoc-acdm', 'Assoc-voc']
test['IsHighEducated']=np.where(test['education'].isin(values), True, False)

#pd.crosstab(train['workclass'], train['>50k'], normalize='index')*100
values=['Federal-gov', 'Self-emp-inc']
test['GovernmentOrBoss']=np.where(test['workclass'].isin(values), True, False)

#pd.crosstab(train['marital-status'], train['>50k'], normalize='index')*100
values=['Divorced', 'Never-married', 'Separated', 'Widowed']
test['Married']=np.where(test['marital-status'].isin(values), False, True)

#pd.crosstab(train['occupation'], train['>50k'], normalize='index')*100
values=['Prof-specialty', 'Exec-managerial']
test['HighPaying']=np.where(test['occupation'].isin(values), True, False)

values=['Asian-Pac-Islander', 'White']
test['AsianOrWhite']=np.where(test['race'].isin(values), True, False)

Europe=['Portugal', 'Italy', 'Germany', 'France', 'Greece', 'Yugoslavia', 'Poland', 'Hungary']
asian=['Japan', 'Vietnam', 'Thailand', 'India', 'Cambodia', 'Iran', 'Taiwan', 'Hong', 'China', 'Philippines']
GB=['England', 'Scotland', 'Ireland']
america=['Mexico', 'Canada', 'United-States']

test['Country']='Others'
test['Country']=np.where(test['native-country'].isin(Europe), 'Europe', test['Country'])
test['Country']=np.where(test['native-country'].isin(asian), 'Asia', test['Country'])
test['Country']=np.where(test['native-country'].isin(GB), 'GB', test['Country'])
test['Country']=np.where(test['native-country'].isin(america), 'North America', test['Country'])

test['profit']=test['capital-gain']-test['capital-loss']
test['HighProfit']=np.where(test['profit']>10000, True, False)

test['AgeBin']='Others'
test['AgeBin']=np.where((test['age']>=0) & (test['age']<=10), 'Child', test['AgeBin'])
test['AgeBin']=np.where((test['age']>10) & (test['age']<=20), 'Teenage', test['AgeBin'])
test['AgeBin']=np.where((test['age']>20) & (test['age']<=30), 'Youth', test['AgeBin'])
test['AgeBin']=np.where((test['age']>30) & (test['age']<=55), 'Adult', test['AgeBin'])
test['AgeBin']=np.where((test['age']>55) & (test['age']<=80), 'Retired', test['AgeBin'])
test['AgeBin']=np.where((test['age']>80) & (test['age']<=100), 'Senile', test['AgeBin'])

features=['education', 'gender', 'IsHighEducated', 'GovernmentOrBoss', 'Married', \
         'HighPaying', 'AsianOrWhite', 'Country', 'HighProfit', 'AgeBin']

for i in features:
    test[i]=test[i].astype('category')

In [390]:
features=['workclass', 'occupation', 'education', 'marital-status', 'gender', 'relationship', 'IsHighEducated', 'GovernmentOrBoss', 'Married', \
         'HighPaying', 'AsianOrWhite', 'Country', 'HighProfit', 'AgeBin']

for i in features:
    test[i]=test[i].astype('category')

testX=test[features]

In [391]:
predictions=model.predict(testX)

In [396]:
output=pd.DataFrame({
    'income_>50K': predictions
})

output.to_csv('/kaggle/working/output.csv')